# Credit card default prediction: final evaluation

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, average_precision_score, brier_score_loss,
    confusion_matrix, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
TRAIN_PATH = "credit_train.csv"
TEST_PATH = "credit_test.csv"
RANDOM_STATE = 42
CV_FOLDS = 5
TARGET = "default"
LOGISTIC_C = 10
FOREST_TREES = 300
FOREST_MAX_DEPTH = 12
FOREST_MIN_SAMPLES_LEAF = 20
CPU_CORES = 8
os.environ["LOKY_MAX_CPU_COUNT"] = str(CPU_CORES)

## 1. Introduction

In this notebook we evaluate the model chosen earlier, a Random Forest, against the logistic regression. We choose the decision threshold, check calibration, measure performance on the test set and analyse which variables matter most. It ends with the conclusions of the whole project.

## 2. Data loading

In [ ]:
credit_train = pd.read_csv(TRAIN_PATH)
credit_test = pd.read_csv(TEST_PATH)

categorical_features = ["SEX", "EDUCATION", "MARRIAGE", "STATUS_1"]
numeric_features = [
    "AGE", "AGE_SQUARED_DEVIATION", "LOG_LIMIT",
    "DELAY_1", "MONTHS_DELAYED", "WORST_DELAY", "MONTHS_PAID_FULL",
    "UTILIZATION_MEAN", "PAYMENT_RATIO_MEAN", "DEBT_TREND",
    "LOG_PAYMENT_LAST", "LOG_PAYMENT_MEAN", "INACTIVE",
]
model_features = categorical_features + numeric_features

X_train, y_train = credit_train[model_features], credit_train[TARGET]
X_test, y_test = credit_test[model_features], credit_test[TARGET]

pd.DataFrame({
    "clients": [len(y_train), len(y_test)],
    "defaults": [y_train.sum(), y_test.sum()],
    "default_rate": [y_train.mean(), y_test.mean()],
}, index=["train", "test"]).round(3)

## 3. Models

We rebuild both models with the hyperparameters chosen in `02_Modeling_CC` and the same preprocessing.

In [ ]:
forest_model = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("categorical", OneHotEncoder(), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ])),
    ("classifier", RandomForestClassifier(
        n_estimators=FOREST_TREES,
        max_depth=FOREST_MAX_DEPTH,
        min_samples_leaf=FOREST_MIN_SAMPLES_LEAF,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

logistic_model = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("categorical", OneHotEncoder(drop="first"), categorical_features),
        ("numeric", StandardScaler(), numeric_features),
    ])),
    ("classifier", LogisticRegression(C=LOGISTIC_C, max_iter=1000)),
])

We compute the out-of-fold predictions on the training set, which we will use to choose the threshold and check calibration, and then train the models on the whole training set.

In [ ]:
cross_validation = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

reference_models = {"Logistic regression": logistic_model, "Random Forest": forest_model}
out_of_fold_probabilities = {}
for model_name, model in reference_models.items():
    probabilities = cross_val_predict(
        model, X_train, y_train, cv=cross_validation, method="predict_proba"
    )
    out_of_fold_probabilities[model_name] = probabilities[:, 1]
    model.fit(X_train, y_train)

The `threshold_metrics` function turns probabilities into decisions (1 = flagged as default risk) and summarises the confusion matrix, precision, recall and F1.

In [ ]:
def threshold_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    true_negatives, false_positives, false_negatives, true_positives = confusion_matrix(
        y_true, predictions
    ).ravel()
    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / (true_positives + false_negatives)
    return {
        "threshold": threshold,
        "% flagged": predictions.mean() * 100,
        "defaults detected (TP)": true_positives,
        "defaults missed (FN)": false_negatives,
        "good clients flagged (FP)": false_positives,
        "precision": precision,
        "recall": recall,
        "F1": 2 * precision * recall / (precision + recall),
    }

## 4. Decision threshold

To turn probabilities into decisions we need a threshold. We use the one that **maximises the F1** of the Random Forest, the balance between precision and recall.

We try thresholds between 0.05 and 0.70 on the out-of-fold training predictions, without touching the test set.

In [ ]:
forest_probabilities = out_of_fold_probabilities["Random Forest"]

thresholds = np.round(np.arange(0.05, 0.71, 0.01), 2)
f1_scores = [
    threshold_metrics(y_train, forest_probabilities, threshold)["F1"]
    for threshold in thresholds
]

f1_by_threshold = pd.Series(f1_scores, index=thresholds)
decision_threshold = f1_by_threshold.idxmax()
decision_threshold

In [ ]:
ax = f1_by_threshold.plot(figsize=(7, 4))
ax.axvline(decision_threshold, color="grey", linestyle="--", label="Chosen threshold")
ax.set_title("F1 by threshold (Random Forest, out-of-fold predictions)")
ax.set_xlabel("Probability threshold")
ax.set_ylabel("F1 of the default class")
ax.legend()
plt.show()

**The threshold that maximises F1 is 0.29**, well below the default 0.5. With it, on the training set, the Random Forest flags 24% of clients, detects 57% of defaults and is right for 53% of the clients it flags (F1 of 0.549).

The curve is flat between 0.25 and 0.35 (F1 from 0.538 to 0.549), so small changes in the threshold barely matter. With the default 0.5, F1 would drop to 0.48.

## 5. Calibration

We need to make sure the probabilities the model outputs are reliable. We group the out-of-fold predictions into 10 bands with the same number of clients and compare the mean predicted probability with the observed default rate.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for model_name in ["Logistic regression", "Random Forest"]:
    observed_rate, predicted_probability = calibration_curve(
        y_train, out_of_fold_probabilities[model_name], n_bins=10, strategy="quantile"
    )
    ax.plot(predicted_probability, observed_rate, marker="o", label=model_name)
ax.plot([0, 1], [0, 1], color="grey", linestyle="--", label="Perfect calibration")
ax.set_title("Calibration curve (out-of-fold predictions)")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed default rate")
ax.legend()
plt.show()

**Both models are well calibrated**: the points follow the diagonal, with deviations of 2 to 3 points at most. No correction is needed.

## 6. Final evaluation on the test set

We use the test set for the first and only time, with the model and threshold already fixed. The logistic regression is included as a reference.

In [ ]:
test_probabilities = {
    "Logistic regression": logistic_model.predict_proba(X_test)[:, 1],
    "Random Forest": forest_model.predict_proba(X_test)[:, 1],
}

test_metrics = pd.DataFrame({
    model_name: {
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "PR-AUC": average_precision_score(y_test, probabilities),
        "Brier score": brier_score_loss(y_test, probabilities),
    }
    for model_name, probabilities in test_probabilities.items()
}).T
test_metrics.round(4)

In [ ]:
test_decisions = pd.DataFrame({
    model_name: threshold_metrics(y_test, probabilities, decision_threshold)
    for model_name, probabilities in test_probabilities.items()
}).T
test_decisions.round(3)

The Random Forest confusion matrix shows the four types of decision on the test set: defaults detected, defaults missed, good clients flagged by mistake and good clients approved.

In [ ]:
test_flagged = (test_probabilities["Random Forest"] >= decision_threshold).astype(int)
display = ConfusionMatrixDisplay.from_predictions(
    y_test, test_flagged, display_labels=["No default", "Default"],
    cmap="Blues", values_format="d",
)
display.ax_.set_title(f"Random Forest confusion matrix (test, threshold {decision_threshold:.2f})")
display.ax_.set_xlabel("Model prediction")
display.ax_.set_ylabel("Actual value")
plt.show()

**The test set confirms the cross-validation results**, with no signs of overfitting:

| Model | ROC-AUC | PR-AUC | Brier score |
|---|---|---|---|
| Logistic regression | 0.764 | 0.526 | 0.139 |
| Random Forest | 0.779 | 0.566 | 0.135 |

Out of the 6,000 test clients and with the 0.29 threshold, the Random Forest:

- detects **741 of the 1,327 defaults (56%)** and misses 586;
- flags **666 of the 4,673 good clients (14%)** by mistake;
- out of every 100 clients flagged, 53 actually default.

Compared with the logistic regression, it detects 58 more defaults at the cost of flagging 86 more good clients (F1 of 0.542 vs 0.527).

In a real setting, the threshold could be tuned according to costs. Which is more expensive, a false positive or a false negative? The threshold could be lowered to detect more defaults at the cost of more rejections.

## 7. Interpretability: permutation importance

For each variable we shuffle its values in the test set and measure how much the Random Forest PR-AUC drops. The bigger the drop, the more the model relies on that variable. It is computed on the 17 original variables, not on the one-hot columns.

In [ ]:
permutation_result = permutation_importance(
    forest_model, X_test, y_test, scoring="average_precision",
    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1,
)
feature_importance = pd.Series(
    permutation_result.importances_mean, index=model_features
).sort_values()

In [ ]:
ax = feature_importance.plot(kind="barh", figsize=(8, 7))
ax.set_title("Permutation importance (Random Forest, test)")
ax.set_xlabel("Drop in PR-AUC when the feature is shuffled")
ax.set_ylabel("Variable")
plt.show()

**Recent payment behaviour is what best predicts default.**

1. **Delay and payment status in September** (`DELAY_1` and `STATUS_1`): the strongest signal.
2. **Delay history** (`MONTHS_DELAYED` and `WORST_DELAY`), followed by payment capacity and credit usage.
3. **Demographic variables: almost irrelevant**, with an importance of 0.003 or less.

These are predictive importances, not causal effects.

The heatmap shows the observed default rate in the training set by two of the most important variables: the months of delay in September (`DELAY_1`, grouping 3 or more) and the number of months with delay in the six-month period (`MONTHS_DELAYED`).

In [ ]:
delay_data = credit_train.assign(DELAY_1=credit_train["DELAY_1"].clip(upper=3))
default_rate_grid = delay_data.pivot_table(
    index="DELAY_1", columns="MONTHS_DELAYED", values=TARGET, aggfunc="mean"
).rename(index={3: "3 or more"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    default_rate_grid, annot=True, fmt=".0%", cmap="Reds", ax=ax,
    cbar_kws={"label": "Default rate"},
)
ax.set_title("Default rate by current delay and delay history")
ax.set_xlabel("Months with delay in the last six months")
ax.set_ylabel("Months of delay in September")
plt.show()

- **With no delay in September**, risk grows with the history: from 12% with no delay at all to 45% with five months of delay.
- **With 2 or more months of delay in September**, default exceeds 55% in almost all cases and reaches 78% with a constant delay history.
- **One month of delay in September (code 1) weighs much less than two:** 25% vs 62% with the same history. This matches what we saw in the EDA: code 1 is unreliable.

The two variables complement each other: the current delay marks the jump in risk and the history grades it. This is the kind of interaction trees capture better than logistic regression.

## 8. Conclusions

### 8.1 How does default vary across demographic variables?

| Variable | Highest default | Lowest default | Cramér's V |
|---|---|---|---|
| Credit limit | ≤ NT$50,000 (31.8%) | > NT$500,000 (11.2%) | 0.164 |
| Education | High school (25.2%) | Postgraduate (19.2%) | 0.073 |
| Age | 21–25 and > 60 (≈ 27%) | 31–35 (19.4%) | 0.062 |
| Sex | Men (24.2%) | Women (20.8%) | 0.040 |
| Marital status | Divorced (26.0%) | Single (20.9%) | 0.034 |

The differences are statistically significant but small, and they almost disappear once the model takes payment behaviour into account.

Note: Cramér's V measures the strength of association between two nominal variables. It ranges from 0 (independence) to 1 (perfect dependence).

### 8.2 Which variables best predict default?

Although it may sound obvious, recent payment behaviour carries the most weight: delay in September, followed by delay history and payment capacity. It is also worth stressing, as answered above, that demographic variables are almost irrelevant. This last point is informative in itself.

### 8.3 Final model

| Model | ROC-AUC test | PR-AUC test | Brier test |
|---|---|---|---|
| Logistic regression | 0.764 | 0.526 | 0.139 |
| Random Forest | **0.779** | **0.566** | **0.135** |

- **Random Forest** is the chosen model: better on every metric and well calibrated, although its advantage over the logistic regression is moderate.
- With the **0.29 threshold** (maximum F1), on the test set it detects **56% of defaults** and flags only **14% of good clients** by mistake. In a real setting, the threshold would be set using the bank's actual costs.